# ASG Airlines: End-to-End Data Engineering Case Study

### Submitted By: Aditi Ahuja , MCA

**Objective** : Build a data pipeline for ASG Airlines to:

1. **Clean the Data:** Fix broken flight IDs, format timestamps, fill missing values, and mask passenger private info (PII).
2. **Fix Overnight Flights:** Calculate true flight durations for flights that land the next day.
3. **Deliver Insights:** Create clean data and a Power BI dashboard showing flight durations, busy routes, delays, and airline trends.

### 1. Data Ingestion 
Ingesting raw data from excel sheet to pandas dataframe

In [172]:
import pandas as pd
import numpy as np 
import re 
import hashlib

excel_load = r"C:\Users\Aditi Ahuja\PLACEMENTS\Airlines Use Case\UseCase - Airlines.xlsx" 

excel_book = pd.ExcelFile(excel_load)

flights = excel_book.parse('flights')
passengers = excel_book.parse('passengers')
bookings = excel_book.parse('bookings')
payments = excel_book.parse('payments')

print(" DATA INGESTION SUCCESSFUL")

print(f"• Flights Loaded:    {len(flights)} records")
print(f"• Passengers Loaded: {len(passengers)} records")
print(f"• Bookings Loaded:   {len(bookings)} records")
print(f"• Payments Loaded:   {len(payments)} records")



 DATA INGESTION SUCCESSFUL
• Flights Loaded:    1020 records
• Passengers Loaded: 1039 records
• Bookings Loaded:   1000 records
• Payments Loaded:   1000 records


### 2. Data Profiling 
Checking each dataset to understand the structure, finding missing and duplicate records.

### 2.1 Flights Cleaning


We clean the Flights dataset by removing only exact duplicate rows, standardizing text fields, filling missing/UNKNOWN airline values where the flight-prefix mapping is supported by the dataset, and converting timestamps into a consistent format. We also calculate flight duration, handle overnight flights, and create a route column.

In [173]:
# Create a copy so the original data remains unchanged
flights_clean = flights.copy()

# 1. Remove only exact duplicate rows
flights_clean = flights_clean.drop_duplicates()

# 2. Standardize text columns
for col in ["flight_id", "airline", "source", "destination"]:
    flights_clean[col] = flights_clean[col].astype("string").str.strip()

# Standardize formats
flights_clean["flight_id"] = flights_clean["flight_id"].str.upper()
flights_clean["airline"] = flights_clean["airline"].str.title()
flights_clean["source"] = flights_clean["source"].str.upper()
flights_clean["destination"] = flights_clean["destination"].str.upper()

# 3. Get flight prefix
flights_clean["flight_prefix"] = flights_clean["flight_id"].str[:2]

# 4. Fill missing / UNKNOWN airline values
# Mapping is based on the patterns present in this dataset
airline_map = {
    "AI": "Air India",
    "6F": "IndiGo",
    "SJ": "SpiceJet",
    "UK": "Vistara"
}

missing_airline = (
    flights_clean["airline"].isna() |
    flights_clean["airline"].str.upper().eq("UNKNOWN")
)

flights_clean.loc[missing_airline, "airline"] = (
    flights_clean.loc[missing_airline, "flight_prefix"].map(airline_map)
)

# 5. Convert departure and arrival to datetime
flights_clean["departure_time"] = pd.to_datetime(
    flights_clean["departure_time"],
    errors="coerce"
)

flights_clean["arrival_time"] = pd.to_datetime(
    flights_clean["arrival_time"],
    errors="coerce"
)

# 6. Calculate flight duration in hours
flights_clean["duration_hours"] = (
    flights_clean["arrival_time"] -
    flights_clean["departure_time"]
).dt.total_seconds() / 3600

# 7. Handle overnight flights
# Overnight means the arrival date is after the departure date
flights_clean["overnight_flag"] = (
    flights_clean["arrival_time"].dt.date >
    flights_clean["departure_time"].dt.date
)

# Fix any negative duration caused by incorrect raw dates
flights_clean.loc[
    flights_clean["duration_hours"] < 0,
    "duration_hours"
] += 24

flights_clean["duration_hours"] = (
    flights_clean["duration_hours"].round(2)
)

# 8. Create route
flights_clean["route"] = (
    flights_clean["source"] + " - " +
    flights_clean["destination"]
)

# 9. View the cleaned data
print("First 5 rows after cleaning:")
display(flights_clean.head())

First 5 rows after cleaning:


,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_prefix,duration_hours,overnight_flag,route
0,SJ010,Spicejet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00,SJ,2.90,True,CCU - MAA
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00,AI,1.80,True,BOM - CCU
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00,UK,1.75,True,BOM - CCU
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00,AI,2.60,True,BOM - CCU
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00,AI,4.98,True,MAA - BOM


**Insights**
1. Exact duplicates: Removed only completely identical rows using drop_duplicates(). The entire duplicate row is removed.
2. Duplicate Flight IDs: We did not remove rows just because flight_id is repeated. If the other flight details are different, both records are retained.
3. Airline values: Missing or UNKNOWN airline values were filled using the flight-prefix pattern found in this dataset. We did not change the existing 6F flight IDs.
4. Date/time: Departure and arrival columns were converted to datetime so duration could be calculated consistently.
5. Overnight handling: If arrival time appeared earlier than departure time, 24 hours was added to the duration. This prevents a valid overnight flight from appearing to have a negative duration.
6. Route: A new route column was created using source - destination.
Original data: flights.copy() keeps the original flights dataset unchanged while we work on flights_clean.

### 2.2 Payments Cleaning & Transformation

For Payments, we will keep all valid records, remove only exact duplicates, convert the amount to numeric, and flag invalid/missing amounts. We will not use the median because this is financial data.

In [174]:
# ==================================
# PAYMENTS CLEANING
# ==================================

# Create working copy
payments_clean = payments.copy()

# 1. Remove only exact duplicate rows
payments_clean = payments_clean.drop_duplicates()

# 2. Standardize text columns
for col in ["payment_id", "booking_id", "payment_method"]:
    payments_clean[col] = (
        payments_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# 3. Convert amount to numeric
# INVALID and missing values become NaN
payments_clean["payment_amount"] = pd.to_numeric(
    payments_clean["amount"],
    errors="coerce"
)

# 4. Flag invalid or missing amounts
payments_clean["amount_issue"] = payments_clean["payment_amount"].isna()

# 5. Check that valid amounts are positive
print("Invalid / Missing Amounts:", payments_clean["amount_issue"].sum())
print(
    "Zero / Negative Amounts:",
    (
        payments_clean["payment_amount"].notna()
        & (payments_clean["payment_amount"] <= 0)
    ).sum()
)

display(payments_clean.head())

Invalid / Missing Amounts: 78
Zero / Negative Amounts: 0


,payment_id,booking_id,amount,payment_method,payment_amount,amount_issue
0,PAY1000,B1116,9883.49,NETBANKING,9883.49,False
1,PAY1001,B1738,8457.96,NETBANKING,8457.96,False
2,PAY1002,B1873,6495.37,UPI,6495.37,False
3,PAY1003,B1914,5079.38,NETBANKING,5079.38,False
4,PAY1004,B1967,12518.31,CARD,12518.31,False


In [175]:
print("Rows after cleaning:", len(payments_clean))
print("Duplicate rows:", payments_clean.duplicated().sum())
print("Missing Payment IDs:", payments_clean["payment_id"].isna().sum())
print("Missing Booking IDs:", payments_clean["booking_id"].isna().sum())
print("Invalid / Missing Amounts:", payments_clean["amount_issue"].sum())
print("Valid Payment Amounts:", payments_clean["payment_amount"].notna().sum())
print("Minimum Valid Amount:", payments_clean["payment_amount"].min())
print("Maximum Valid Amount:", payments_clean["payment_amount"].max())

Rows after cleaning: 1000
Duplicate rows: 0
Missing Payment IDs: 0
Missing Booking IDs: 0
Invalid / Missing Amounts: 78
Valid Payment Amounts: 922
Minimum Valid Amount: 1002.59
Maximum Valid Amount: 14992.95


**Insights:**

1. Exact duplicates: Removed only completely identical rows using drop_duplicates(). No exact duplicate payment rows were found.
2. Duplicate Payment IDs: Payment IDs were checked for duplicates. No duplicate payment IDs were found.
3. Payment Amount: The original amount column was converted to a numeric payment_amount column. Values such as INVALID and missing values become NaN.
4. Missing / Invalid Amounts: 48 amounts were missing and 30 were marked INVALID, giving 78 problematic payment amounts. These were not replaced with the median because payment amounts are financial values and should not be artificially estimated.
5. Amount Validation: Valid numeric payment amounts were checked to ensure they are positive. No zero or negative payment amounts were found.
6. Payment Method: payment_method was standardized to uppercase and checked against the valid values UPI, CARD, and NETBANKING. No invalid payment methods were found.
7. Data Quality Flag: An amount_issue column was created to identify records where the payment amount is missing or invalid. This allows the issue to be tracked without deleting the payment record.
8. Original Data: payments.copy() keeps the original payments dataset unchanged while we work on payments_clean.
9. Power BI: payment_amount should be used for financial calculations. Missing/invalid amounts can be shown separately as a data-quality issue rather than being filled with an artificial value.

### 2.3 Bookings Cleaning & PII Masking

The bookings data contains booking, passenger, and flight references, along with booking status and some passenger-related PII. We will remove only exact duplicate rows, standardize the identifier/status fields, and handle missing or invalid statuses without deleting the bookings.

In [176]:
# ==================================
# BOOKINGS CLEANING
# ==================================

# Create working copy
bookings_clean = bookings.copy()

# 1. Remove only exact duplicate rows
bookings_clean = bookings_clean.drop_duplicates()

# 2. Standardize important text columns
for col in ["booking_id", "passenger_id", "flight_id", "status"]:
    bookings_clean[col] = (
        bookings_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# 3. Create clean booking status
valid_status = ["CONFIRMED", "CANCELLED", "PENDING"]

bookings_clean["booking_status"] = bookings_clean["status"].where(
    bookings_clean["status"].isin(valid_status),
    "UNKNOWN"
)

display(bookings_clean.head())

,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone,booking_status
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128,CANCELLED
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662,CANCELLED
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220,CANCELLED
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839,CONFIRMED
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266,PENDING


In [177]:
# ==================================
# BOOKINGS VALIDATION
# ==================================

print("Rows after cleaning:", len(bookings_clean))
print("Duplicate rows:", bookings_clean.duplicated().sum())

print("Missing Booking IDs:", bookings_clean["booking_id"].isna().sum())
print("Missing Passenger IDs:", bookings_clean["passenger_id"].isna().sum())
print("Missing Flight IDs:", bookings_clean["flight_id"].isna().sum())

print("\nOriginal Status Values:")
print(bookings_clean["status"].value_counts(dropna=False))

print("\nClean Booking Status:")
print(bookings_clean["booking_status"].value_counts(dropna=False))

Rows after cleaning: 1000
Duplicate rows: 0
Missing Booking IDs: 0
Missing Passenger IDs: 0
Missing Flight IDs: 0

Original Status Values:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
<NA>          45
INVALID       30
Name: count, dtype: Int64

Clean Booking Status:
booking_status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       75
Name: count, dtype: Int64


In [178]:
# Keep the cleaned status and remove the original status column
bookings_clean = bookings_clean.drop(columns=["status"])

display(bookings_clean.head())

,booking_id,passenger_id,flight_id,booking_date,passport_number,seat_number,emergency_contact_name,emergency_contact_phone,booking_status
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,P1945887,3D,Isaac Bakshi,+91-6478475128,CANCELLED
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,L3482012,18A,Anvi Konda,+91-6647078662,CANCELLED
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,G8507659,30C,Udant Dewan,+91-8405938220,CANCELLED
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,M0891776,33A,Harsh Chahal,+91-6264636839,CONFIRMED
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,N5742231,25C,Pahal Balay,+91-9336478266,PENDING


In [179]:
# ==================================
# BOOKING PII PROTECTION
# ==================================

bookings_masked = bookings_clean.copy()

# Mask passport number - keep last 4 characters
bookings_masked["passport_number"] = (
    bookings_masked["passport_number"]
    .astype("string")
    .str[-4:]
    .radd("****")
)

# Mask emergency contact name
bookings_masked["emergency_contact_name"] = (
    bookings_masked["emergency_contact_name"]
    .astype("string")
    .str[0]
    .fillna("")
    + "***"
)

# Mask emergency contact phone - keep last 4 digits
bookings_masked["emergency_contact_phone"] = (
    bookings_masked["emergency_contact_phone"]
    .astype("string")
    .str[-4:]
    .radd("******")
)

display(
    bookings_masked[
        [
            "booking_id",
            "passport_number",
            "emergency_contact_name",
            "emergency_contact_phone"
        ]
    ].head()
)

,booking_id,passport_number,emergency_contact_name,emergency_contact_phone
0,B1000,****5887,I***,******5128
1,B1001,****2012,A***,******8662
2,B1002,****7659,U***,******8220
3,B1003,****1776,H***,******6839
4,B1004,****2231,P***,******8266


**Insights:**

1. Exact duplicates: Removed only completely identical rows using drop_duplicates(). No exact duplicate booking rows were found.
2. Duplicate Booking IDs: Booking IDs were checked for duplicates. We do not remove a record simply because an identifier is repeated without investigating the related data.
3. Booking IDs: No missing booking IDs were found. The booking ID is an important key for connecting bookings with payments.
4. Passenger and Flight IDs: No missing passenger or flight references were found. These fields are needed to connect bookings with the passenger and flight datasets.
5. Booking Status: The original status values were standardized to uppercase. Valid values are CONFIRMED, CANCELLED, and PENDING.
6. Missing / Invalid Status: 45 statuses were missing and 30 were INVALID. These 75 bookings were not deleted. They were represented as UNKNOWN in the cleaned booking_status column.
7. Missing values: A missing non-key value does not automatically mean that the entire booking should be removed. We keep the valid booking and handle the problematic field separately.
8. PII: Passport number, emergency contact name, and emergency contact phone are sensitive fields and are not required for the business KPIs. They should be excluded from the final reporting dataset/Power BI model.
9. Original data: bookings.copy() keeps the original bookings dataset unchanged while we work on bookings_clean.
10. Temporary columns: We remove columns that were only needed for cleaning or validation. The final dataset should contain only useful business/reporting fields.

### 2.4 Passenger Data Cleaning and PII Protection

The passenger dataset contains both analytical fields and personal information. We will clean the passenger records first, keep valid passenger IDs for relationships, and create a separate reporting dataset that excludes PII not required for flight analysis.

In [180]:
# ==================================
# PASSENGERS CLEANING
# ==================================

# Create working copy
passengers_clean = passengers.copy()

# 1. Remove only exact duplicate rows
passengers_clean = passengers_clean.drop_duplicates()

# 2. Standardize text columns
for col in [
    "passenger_id",
    "first_name",
    "last_name",
    "gender",
    "email",
    "phone",
    "aadhaar_id"
]:
    passengers_clean[col] = (
        passengers_clean[col]
        .astype("string")
        .str.strip()
    )

# 3. Standardize passenger ID
passengers_clean["passenger_id"] = (
    passengers_clean["passenger_id"]
    .str.upper()
)

display(passengers_clean.head())

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


In [181]:
# ==================================
# PASSENGERS VALIDATION
# ==================================

print("Rows after cleaning:", len(passengers_clean))
print("Duplicate rows:", passengers_clean.duplicated().sum())

print("Missing Passenger IDs:",
      passengers_clean["passenger_id"].isna().sum())

print("Duplicate Passenger IDs:",
      passengers_clean["passenger_id"].duplicated().sum())

print("Missing First Names:",
      passengers_clean["first_name"].isna().sum())

print("Missing Last Names:",
      passengers_clean["last_name"].isna().sum())

print("Missing Gender:",
      passengers_clean["gender"].isna().sum())

Rows after cleaning: 1039
Duplicate rows: 0
Missing Passenger IDs: 0
Duplicate Passenger IDs: 39
Missing First Names: 0
Missing Last Names: 10
Missing Gender: 0


In [182]:
pii_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "aadhaar_id",
]

print("PII columns:")
print(pii_columns)

PII columns:
['first_name', 'last_name', 'email', 'phone', 'aadhaar_id']


In [183]:
# ==================================
# PII MASKING
# ==================================

passengers_masked = passengers_clean.copy()

# Mask phone number
passengers_masked["phone"] = (
    passengers_masked["phone"].str[-4:].radd("******")
)

# Mask Aadhaar ID
passengers_masked["aadhaar_id"] = (
    passengers_masked["aadhaar_id"].str[-4:].radd("********")
)

# Mask email
passengers_masked["email"] = (
    passengers_masked["email"].str[0].fillna("") + "***"
)

display(
    passengers_masked[
        ["passenger_id", "email", "phone", "aadhaar_id"]
    ].head()
)

,passenger_id,email,phone,aadhaar_id
0,P1000,v***,******3790,********6001
1,P1001,k***,******2297,********2654
2,P1002,m***,******5092,********8161
3,P1003,m***,******7151,********6475
4,P1004,s***,******5113,********6483


In [184]:
# ==================================
# PASSENGER REPORTING DATA
# ==================================

passengers_reporting = passengers_clean[
    [
        "passenger_id",
        "age",
        "gender"
    ]
].copy()

display(passengers_reporting.head())

,passenger_id,age,gender
0,P1000,52,F
1,P1001,15,M
2,P1002,72,M
3,P1003,61,F
4,P1004,21,M


**Insights:**

1. Exact duplicates: Removed only completely identical passenger records using drop_duplicates(). No exact duplicate passenger rows were found.
2. Passenger ID: passenger_id is an important key for connecting passengers with bookings. No missing or duplicate passenger IDs were found.
3. Missing Last Name: 10 passenger records have a missing last_name. We kept these records because the passenger ID is still available. We did not invent or replace the missing name.
4. PII Identification: The dataset contains sensitive information including names, email addresses, phone numbers, Aadhaar IDs, and date of birth.
5. PII Masking: Sensitive fields such as phone number, Aadhaar ID, and email were masked in a separate passengers_masked dataset so that the original values are not exposed for general use.
6. Reporting Dataset: PII that is not required for operational analysis was excluded from passengers_reporting. Only passenger_id, age, and gender are retained for reporting.
7. Privacy and Access Control: Raw data containing original PII should be accessible only to authorized users. The Power BI/reporting layer uses the non-sensitive reporting dataset to avoid unnecessary exposure of passenger information.
8. Original Data: passengers.copy() keeps the original passenger dataset unchanged while cleaning and protecting the data.
9. Data Minimization: Only fields required for the business analysis are carried forward to the reporting layer. This reduces unnecessary exposure of personal information.

### 3. Data Modelling, Storage and Business KPIs

### 3.1 Average Flight Duration
Interpretation:
We already calculated duration_hours while cleaning the Flights dataset. Now we simply calculate its average for the KPI.

In [185]:
# Average Flight Duration

average_flight_duration = flights_clean["duration_hours"].mean()

print("Average Flight Duration:", round(average_flight_duration, 2), "hours")

Average Flight Duration: 2.74 hours


### 3.2 Route-wise Traffic

For route-wise traffic, we count how many flights operate on each route. The route column was already created during the Flights transformation.

In [186]:
# Route-wise Traffic

route_traffic = (
    flights_clean
    .groupby("route")
    .size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

display(route_traffic.head(10))

,route,flight_count
6,BOM - CCU,90
12,CCU - DEL,72
25,MAA - BLR,65
0,BLR - BOM,60
24,HYD - MAA,57
18,DEL - HYD,54
23,HYD - DEL,42
7,BOM - DEL,39
11,CCU - BOM,33
15,DEL - BLR,29


### 3.3 Distribution of Flights by Airline

Here we count the number of flights for each airline.

In [187]:
# Standardize airline names

airline_names = {
    "air india": "Air India",
    "indigo": "IndiGo",
    "spicejet": "SpiceJet",
    "vistara": "Vistara"
}

flights_clean["airline"] = (
    flights_clean["airline"]
    .str.strip()
    .str.lower()
    .map(airline_names)
)

In [188]:
# Distribution of Flights by Airline

airline_distribution = (
    flights_clean
    .groupby("airline")
    .size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)
# Add total count
total_flights = airline_distribution["flight_count"].sum()

airline_distribution.loc[len(airline_distribution)] = [
    "Total",
    total_flights
]

display(airline_distribution)



,airline,flight_count
1,IndiGo,273
0,Air India,255
2,SpiceJet,247
3,Vistara,230
4,Total,1005


### 3.4 Delays / Anomalies

There is an important limitation in this dataset: there are no scheduled departure/arrival times and actual departure/arrival times separately. Therefore, we should not invent a "delay in minutes" calculation.
Instead, we can report the actual anomalies that exist in the data, such as overnight flights and invalid/missing values.


In [189]:
# Flight Anomalies

overnight_flights = flights_clean["overnight_flag"].sum()

duration_issues = (
    flights_clean["duration_hours"] <= 0
).sum()

missing_airline = flights_clean["airline"].isna().sum()

print("Overnight Flights:", overnight_flights)
print("Invalid Duration Records:", duration_issues)
print("Missing Airline Records:", missing_airline)

Overnight Flights: 122
Invalid Duration Records: 0
Missing Airline Records: 0


In [190]:
# Booking Anomalies

unknown_booking_status = (
    bookings_clean["booking_status"] == "UNKNOWN"
).sum()

print("Unknown Booking Status:", unknown_booking_status)

Unknown Booking Status: 75


In [191]:
# Payment Anomalies

invalid_payment_amounts = payments_clean["amount_issue"].sum()

print("Invalid / Missing Payment Amounts:",
      invalid_payment_amounts)

Invalid / Missing Payment Amounts: 78


### 3.5 Additional KPIs

The assignment also asks for additional useful KPIs. We can calculate a few simple ones.

In [192]:
total_flights = len(flights_clean)

print("Total Flights:", total_flights)

Total Flights: 1005


In [193]:
total_bookings = len(bookings_clean)

print("Total Bookings:", total_bookings)

Total Bookings: 1000


In [194]:
total_passengers = len(passengers_clean)

print("Total Passengers:", total_passengers)

Total Passengers: 1039


In [195]:
booking_status_distribution = (
    bookings_clean["booking_status"]
    .value_counts()
    .reset_index()
)

booking_status_distribution.columns = [
    "booking_status",
    "booking_count"
]

display(booking_status_distribution)

,booking_status,booking_count
0,CONFIRMED,320
1,CANCELLED,314
2,PENDING,291
3,UNKNOWN,75


In [196]:
total_payment_amount = payments_clean["payment_amount"].sum()

print("Total Valid Payment Amount:",
      round(total_payment_amount, 2))

Total Valid Payment Amount: 7385142.98


In [197]:
average_payment_amount = payments_clean["payment_amount"].mean()

print("Average Payment Amount:",
      round(average_payment_amount, 2))

Average Payment Amount: 8009.92


### 3.6 Data Modelling

The four cleaned datasets are connected through their ID columns. We need to document these relationships before moving to Power BI.

In [198]:
# Passenger → Booking relationship

invalid_passengers = (
    ~bookings_clean["passenger_id"]
    .isin(passengers_clean["passenger_id"])
).sum()

print("Invalid Passenger References:", invalid_passengers)

Invalid Passenger References: 0


In [199]:
# Flight → Booking relationship

invalid_flights = (
    ~bookings_clean["flight_id"]
    .isin(flights_clean["flight_id"])
).sum()

print("Invalid Flight References:", invalid_flights)

Invalid Flight References: 0


In [200]:
# Booking → Payment relationship

invalid_bookings = (
    ~payments_clean["booking_id"]
    .isin(bookings_clean["booking_id"])
).sum()

print("Invalid Booking References:", invalid_bookings)

Invalid Booking References: 0


### 3.7 Storage / Export

We need to save the cleaned datasets so they can be used by Power BI.

In [201]:
# Export cleaned datasets

flights_clean.to_csv("flights_clean.csv", index=False)
passengers_clean.to_csv("passengers_clean.csv", index=False)
bookings_clean.to_csv("bookings_clean.csv", index=False)
payments_clean.to_csv("payments_clean.csv", index=False)

print("Cleaned datasets exported successfully.")

Cleaned datasets exported successfully.


In [202]:
import os

print(os.path.exists("flights_clean.csv"))
print(os.path.exists("passengers_clean.csv"))
print(os.path.exists("bookings_clean.csv"))
print(os.path.exists("payments_clean.csv"))

True
True
True
True


### 3.8 Prepare Data for Power BI

The purpose here is simply to create reporting tables containing the fields actually needed for analysis. PII and temporary cleaning columns are excluded from the Power BI layer.

In [203]:
# Final Flights table for Power BI

flights_powerbi = flights_clean[
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "duration_hours",
        "overnight_flag",
        "route"
    ]
].copy()

In [204]:
# Final Bookings table for Power BI

bookings_powerbi = bookings_clean[
    [
        "booking_id",
        "passenger_id",
        "flight_id",
        "booking_date",
        "seat_number",
        "booking_status"
    ]
].copy()


In [205]:
# Final Passengers table for Power BI

passengers_powerbi = passengers_clean[
    [
        "passenger_id",
        "age",
        "gender"
    ]
].copy()

# Keep one record for each passenger ID
passengers_powerbi = passengers_powerbi.drop_duplicates(
    subset=["passenger_id"]
)

In [206]:
# Final Payments table for Power BI

payments_powerbi = payments_clean[
    [
        "payment_id",
        "booking_id",
        "payment_amount",
        "payment_method"
    ]
].copy()

In [207]:
print("Flights:", flights_powerbi.shape)
print("Bookings:", bookings_powerbi.shape)
print("Passengers:", passengers_powerbi.shape)
print("Payments:", payments_powerbi.shape)

Flights: (1005, 9)
Bookings: (1000, 6)
Passengers: (1000, 3)
Payments: (1000, 4)


In [208]:
print("Duplicate Flight Rows:", flights_powerbi.duplicated().sum())
print("Duplicate Booking Rows:", bookings_powerbi.duplicated().sum())
print("Duplicate Passenger Rows:", passengers_powerbi.duplicated().sum())
print("Duplicate Payment Rows:", payments_powerbi.duplicated().sum())

Duplicate Flight Rows: 0
Duplicate Booking Rows: 0
Duplicate Passenger Rows: 0
Duplicate Payment Rows: 0


In [209]:
print(
    "Invalid Passenger References:",
    (~bookings_powerbi["passenger_id"]
     .isin(passengers_powerbi["passenger_id"])).sum()
)

print(
    "Invalid Flight References:",
    (~bookings_powerbi["flight_id"]
     .isin(flights_powerbi["flight_id"])).sum()
)

print(
    "Invalid Booking References:",
    (~payments_powerbi["booking_id"]
     .isin(bookings_powerbi["booking_id"])).sum()
)

Invalid Passenger References: 0
Invalid Flight References: 0
Invalid Booking References: 0


In [210]:
print("Passenger Rows:", len(passengers_powerbi))
print(
    "Unique Passenger IDs:",
    passengers_powerbi["passenger_id"].nunique()
)

Passenger Rows: 1000
Unique Passenger IDs: 1000


In [211]:
bookings_masked.to_csv(
    "bookings_masked.csv",
    index=False
)

print("Masked booking dataset exported successfully.")

Masked booking dataset exported successfully.


In [212]:
passengers_masked.to_csv(
    "passengers_masked.csv",
    index=False
)

print("Masked passenger dataset exported successfully.")

Masked passenger dataset exported successfully.


In [213]:
# Export final Power BI reporting tables

flights_powerbi.to_csv("flights_powerbi.csv", index=False)
bookings_powerbi.to_csv("bookings_powerbi.csv", index=False)
passengers_powerbi.to_csv("passengers_powerbi.csv", index=False)
payments_powerbi.to_csv("payments_powerbi.csv", index=False)

print("Power BI files exported successfully.")

Power BI files exported successfully.
